In [ ]:
import pandas as pd
import missingno as msno

# 0.1 - Process RegBL

This notebook processes the RegBL dataset to make it usable for Eval2Rn as a basis of building entities.

## Load the data

In [ ]:
ROOT = "../../"
DATA = f"{ROOT}data/raw/03_OFS_regbl/"
BUILDINGS = f"{DATA}gebaeude_batiment_edificio.csv"
CODES = f"{DATA}kodes_codes_codici.csv"

In [ ]:
df = pd.read_csv(BUILDINGS, sep='\t')

In [ ]:
df.head()

In [ ]:
df['EGID'].isna().sum()

In [ ]:
msno.matrix(df)

## Select variables

In [ ]:
df.columns

We selected the following columns

In [ ]:
cols = [
    'EGID', 'GDEKT', 
    #'GGDENR', 
    'GGDENAME', 
    #'EGRID', 
    #'LGBKR', 
    'LPARZ',
    #'LPARZSX', 
    #'LTYP', 
    'GEBNR', 
    'GBEZ', 
    'GKODE', 
    'GKODN', 
    'GKSCE', 
    'GSTAT',
    'GKAT', 
    'GKLAS', 
    'GBAUJ', 
    'GBAUM', 
    'GBAUP', 
    'GABBJ', 
    'GAREA', 
    'GVOL',
    #'GVOLNORM', 
    #'GVOLSCE', 
    'GASTW', 
    'GANZWHG', 
    'GAZZI', 
    'GSCHUTZR', 
    'GEBF',
    'GWAERZH1', 
    'GENH1', 
    #'GWAERSCEH1', 
    #'GWAERDATH1', 
    'GWAERZH2', 
    'GENH2',
    #'GWAERSCEH2', 
    #'GWAERDATH2', 
    'GWAERZW1', 
    'GENW1', 
    #'GWAERSCEW1',
    #'GWAERDATW1', 
    'GWAERZW2', 
    'GENW2', 
    #'GWAERSCEW2', 
    #'GWAERDATW2',
    #'GEXPDAT'
]

In [ ]:
msno.matrix(df[cols])

## Replace the codes with words

In [ ]:
df_codes = pd.read_csv(CODES, sep='\t')

In [ ]:
# create a list of the columns which have a translation code in the dataset df_codes
categorical_need_translation = list(df_codes['CMERKM'].unique())
categorical_need_translation

In [ ]:
# select the ones that are in my dataframe df_building
categorical_need_translation_building = [col for col in df.columns if col in categorical_need_translation]
categorical_need_translation_building

In [ ]:
for col in categorical_need_translation_building: 
    # filter the rows in df_codes where 'CMERKM' is the column of interest
    col_mapping_df = df_codes[df_codes["CMERKM"] == col]

    # create a dictionary for that column, the keys are the codes 'CECODID', and the values are their meanings, in the long French version 'CODTXTLF'
    mapping_dict = col_mapping_df.set_index("CECODID")["CODTXTLF"].to_dict()

    # replace the codes in df_building[col] with their meanings using the dictionary created above
    df[col] = df[col].map(mapping_dict)

## Transform dates in multiple columns

In [ ]:
date_features = ['GWAERDATH1','GWAERDATW1']

In [ ]:
def transform_date(df, col):
    df[col] = pd.to_datetime(df[col], errors='coerce')
    # Extract numerical features
    df[f"{col}_year"] = df[col].dt.year
    df[f"{col}_month"] = df[col].dt.month
    df[f"{col}_day"] = df[col].dt.day
    df = df.drop(columns=col)
    return df

In [ ]:
for col in date_features:
    df = transform_date(df, col)

## Select only desired columns

In [ ]:
df = df[cols]

## Transform columns names with descriptive names

In [ ]:
codes = {
    #"EGID": "Identificateur fédéral de bâtiment", # Don't rename the EGID as it is our shared key
    "GDEKT": "Canton",
    "GGDENR": "Numéro OFS de la commune",
    "GGDENAME": "Nom de la commune",
    "EGRID": "Identificateur fédéral d'immeuble (bien-fonds)",
    "LGBKR": "N° de secteur du registre foncier",
    "LPARZ": "Numéro d’immeuble",
    "LPARZSX": "Suffixe du numéro d’immeuble",
    "LTYP": "Type d'immeuble",
    "GEBNR": "Numéro officiel du bâtiment",
    "GBEZ": "Désignation du bâtiment",
    "GKODE": "Coordonnée E",
    "GKODN": "Coordonnée N",
    "GKSCE": "Provenance des coordonnées",
    "GSTAT": "Statut du bâtiment",
    "GKAT": "Catégorie de bâtiment",
    "GKLAS": "Classe de bâtiment",
    "GBAUJ": "Année de construction du bâtiment",
    "GBAUM": "Mois de construction du bâtiment",
    "GBAUP": "Période de construction",
    "GABBJ": "Année de démolition du bâtiment",
    "GAREA": "Surface du bâtiment",
    "GVOL": "Volume du bâtiment",
    "GVOLNORM": "Volume du bâtiment : norme",
    "GVOLSCE": "Volume du bâtiment : indication sur la donnée",
    "GASTW": "Nombre de niveaux",
    "GANZWHG": "Nombre de logements",
    "GAZZI": "Nombre de pièces d’hab. indép.",
    "GSCHUTZR": "Abri de protection civile",
    "GEBF": "Surface de référence énergétique",
    "GWAERZH1": "Générateur de chaleur pour le chauffage 1",
    "GENH1": "Source d’énergie / de chaleur pour le chauffage 1",
    "GWAERSCEH1": "Source d‘information pour le chauffage 1",

    # GWAERDATH1
    "GWAERDATH1": "Date de mise à jour pour le chauffage 1",
    "GWAERDATH1_year": "Année de mise à jour pour le chauffage 1",
    "GWAERDATH1_month": "Mois de mise à jour pour le chauffage 1",
    "GWAERDATH1_day": "Jour de mise à jour pour le chauffage 1",
    
    "GWAERZH2": "Générateur de chaleur pour le chauffage 2",
    "GENH2": "Source d’énergie / de chaleur pour le chauffage 2",
    "GWAERSCEH2": "Source d‘information pour le chauffage 2",

    # GWAERDATH2
    "GWAERDATH2": "Date de mise à jour pour le chauffage 2",
    "GWAERDATH2_year": "Année de mise à jour pour le chauffage 2",
    "GWAERDATH2_month": "Mois de mise à jour pour le chauffage 2",
    "GWAERDATH2_day": "Jour de mise à jour pour le chauffage 2",
    
    "GWAERZW1": "Générateur de chaleur pour l'eau chaude 1",
    "GENW1": "Source d’énergie / de chaleur pour l'eau chaude 1",
    "GWAERSCEW1": "Source d‘information pour l'eau chaude 1",

    # GWAERDATW1
    "GWAERDATW1": "Date de mise à jour pour l'eau chaude 1",
    "GWAERDATW1_year": "Année de mise à jour pour l'eau chaude 1",
    "GWAERDATW1_month": "Mois de mise à jour pour l'eau chaude 1",
    "GWAERDATW1_day": "Jour de mise à jour pour l'eau chaude 1",
    
    "GWAERZW2": "Générateur de chaleur pour l'eau chaude 2",
    "GENW2": "Source d’énergie / de chaleur pour l'eau chaude 2",
    "GWAERSCEW2": "Source d‘information pour l'eau chaude 2",

    # GWAERDATW2
    "GWAERDATW2": "Date de mise à jour pour l'eau chaude 2",
    "GWAERDATW2_year": "Année de mise à jour pour l'eau chaude 2",
    "GWAERDATW2_month": "Mois de mise à jour pour l'eau chaude 2",
    "GWAERDATW2_day": "Jour de mise à jour pour l'eau chaude 2",
    
    "GEXPDAT": "Date de l'export"
}

### Add "RegBL -" to column names for nomenclature

In [ ]:
codes = {
    k: (f"RegBL - {v}" if v not in ["Coordonnée E", "Coordonnée N"] else v)
    for k, v in codes.items()
}
# Rename the columns
df.rename(columns=codes, inplace=True)

df.head()

## Save dataset

In [ ]:
df.to_csv(f"{ROOT}data/processed/03_OFS_regbl.csv")